# VibeCI — Technical Competitive Intelligence Agent
### Kaggle Vibe Coding Capstone | Track: Agents for Business

**VibeCI** is a deployable, multi-agent competitive intelligence system built on the **`google.antigravity`** SDK and **Gemini 2.0 Flash**.

It reads a competitor's technical documentation (live, or preloaded mock specs), extracts what their products *actually do*, and contrasts that against their marketing claims — producing structured Battle Cards, Objection Handlers, and Sales Landmines, with **every finding grounded to a clickable source line** in the competitor's own docs. That grounding isn't asserted — it's **scored per finding and gated by an eval** (see the last cell).

A five-agent pipeline (**Strategy → Discovery → Technical Analysis ★ → Synthesis → Fact-Checking**) runs over an **MCP tool server**, and the browser renders the run as a live agent timeline. The Strategy agent reads *your* business context (messaging pillars · roadmap · ICP) and sets the research brief that directs the rest.

---

## Setup Instructions

1. **(Optional) Add a Gemini API key** to Kaggle Secrets (Add-ons → Secrets) with the name `GEMINI_API_KEY` — needed **only** for Live mode. **Demo mode runs with no key.**
2. **Enable internet access** for this notebook (Settings → Internet on) — needed to install packages and open the public tunnel.
3. **Run all cells** — a public URL will appear at the end. The app opens in **Demo mode** (no key required); the toggle in the UI switches to Live mode.

> Demo mode is the canonical showcase: deterministic, no API key, never rate-limits. Live mode runs the real Gemini pipeline and needs a **quota-enabled** key (the Gemini free tier may be `0` for `gemini-2.0-flash`).


In [ ]:
# Cell 1: Load API Key from Kaggle Secrets (optional — Live mode only)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
    print('GEMINI_API_KEY loaded — Live mode available.')
except Exception:
    if os.getenv('GEMINI_API_KEY'):
        print('GEMINI_API_KEY loaded from environment — Live mode available.')
    else:
        print('No GEMINI_API_KEY found — running in Demo mode only (no key required).')


In [ ]:
# Cell 2: Install Dependencies
import subprocess
subprocess.run(['pip', 'install', '-q',
    'google-antigravity>=0.1.4', 'fastapi>=0.110.0', 'uvicorn>=0.28.0',
    'pydantic>=2.0.0', 'httpx>=0.27.0', 'python-dotenv>=1.0.0', 'jinja2>=3.0.0',
    'python-multipart', 'mcp>=1.0.0', 'pyngrok'
], check=True)
print('Dependencies installed')


In [ ]:
# Cell 3: Clone the VibeCI repository
import subprocess, os
os.makedirs('/kaggle/working/vibeci', exist_ok=True)
result = subprocess.run(
    ['git', 'clone', '--depth=1', 'https://github.com/msdanyg/vibeci.git', '/kaggle/working/vibeci'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('Clone failed:', result.stderr)
else:
    print('Repository cloned successfully')
    os.chdir('/kaggle/working/vibeci')

In [ ]:
# Cell 4: Start Server + Open Public URL
import subprocess, time, os
from pyngrok import ngrok

os.chdir('/kaggle/working/vibeci')  # must run from the repo root
PORT = 8080

server = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(4)
print(f'Server started (PID {server.pid})')

tunnel = ngrok.connect(PORT)
print()
print('=' * 60)
print('VibeCI is LIVE at:')
print(f'  {tunnel.public_url}')
print('=' * 60)
print()
print('Demo mode (default) = the full UI + agent run timeline from pre-canned')
print('  reports — no API key, deterministic, instant.')
print('Toggle to Live mode to run the real five-agent Gemini pipeline')
print('  (needs a quota-enabled GEMINI_API_KEY).')


In [ ]:
# Cell 5: Smoke Test (Demo mode — no key required)
import requests, time

BASE = f'http://localhost:{PORT}'
payload = {
    'competitor_name': 'Teramind',
    'doc_url': 'mock://teramind-kb.org/api/v4/specs',
    'marketing_claims': 'Real-time session monitoring with zero-latency keystroke logging.',
    'own_positioning': 'Privacy-first productivity insights without screen recording.',
    'demo_mode': True,
}
job_id = requests.post(f'{BASE}/api/analyze', json=payload).json()['job_id']
print(f'Job started: {job_id}')

for _ in range(60):
    time.sleep(1)
    s = requests.get(f'{BASE}/api/job/{job_id}').json()
    if s['status'] == 'completed':
        r = s['result']
        print(f'\nAnalysis complete for {r["competitor_name"]}')
        print(f'  Gaps: {len(r["gaps"])} | Landmines: {len(r["sales_landmines"])}')
        print(f'  Pipeline events streamed: {len(s.get("events", []))}')
        print(f'  Top finding: {r["key_takeaways"][0][:100]}')
        break
    elif s['status'] == 'failed':
        print(f'Failed: {s["error"]}')
        break
    else:
        print(f'  Running... {len(s.get("events", []))} events streamed')


---

## Eval: every finding is grounded — and we *measure* it

VibeCI's core claim is that **every finding traces to a real line in the competitor's own docs**. The cell below turns that from a slogan into a gated metric. It runs the real demo pipeline for each competitor and scores, per gap, how strongly its best source line backs the finding (a 0–1 **grounding-confidence**: the fraction of the finding's distinctive, IDF- and number-weighted tokens that appear in that line). The same `app/grounding.py` code computes the number the UI shows on each gap card — so the page and the eval never disagree.

The harness **fails (non-zero exit)** if any gap falls below threshold (`GROUND_THRESHOLD = 0.45`), and it's also wired as a `pytest` gate (`tests/test_grounding_eval.py`), so a future change that quietly weakens grounding breaks CI.


In [ ]:
# Cell 6: Grounding-confidence eval (runs the real pipeline, gates on the score)
import subprocess, sys, os

os.chdir('/kaggle/working/vibeci')  # must run from the repo root
result = subprocess.run(
    [sys.executable, '-m', 'eval.grounding_eval'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit('Grounding eval FAILED — a finding is no longer grounded above threshold.')
print('Grounding eval passed — every finding is grounded to a real source line.')
